# Interactive Generation with Your Pretrained GPT

Use this notebook to load a pretrained (or SFT) GPT checkpoint from this repo and generate text.

- Adjust the checkpoint path and model hyperparameters to match your training run
- Tokenizer is set up with the special tokens used in this project
- Generation is simple and fast; for longer/creative outputs, increase `max_new_tokens`

Tip: If you fine-tuned the model (SFT), you can point `checkpoint_path` to an SFT checkpoint instead.


In [1]:
from IPython.display import HTML, display

    def set_css():
      display(HTML('''
      <style>
        pre {
          white-space: pre-wrap;
        }
      </style>
      '''))

    get_ipython().events.register('pre_run_cell', set_css)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/gpt_model
!cp /content/drive/MyDrive/checkpoint_epoch_1.pt /content/drive/MyDrive/gpt_model


Mounted at /content/drive


In [3]:
import torch
import os

src = "/content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pt"
dst = "/content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pth"

print("Loading:", src)
ckpt = torch.load(src, map_location="cpu")

if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    print("Found 'model_state_dict' in checkpoint, unwrapping it.")
else:
    state_dict = ckpt
    print("No 'model_state_dict' key, treating checkpoint as raw state_dict.")

unwanted_prefix = "_orig_mod."
fixed_any = False
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
        fixed_any = True
if fixed_any:
    print("Stripped '_orig_mod.' prefixes from state_dict keys.")

if any(k.startswith("embbeding.") for k in state_dict.keys()):
    print("Found 'embbeding.' typo in keys, fixing to 'embedding.'...")
    corrected = {}
    for k, v in state_dict.items():
        corrected[k.replace("embbeding.", "embedding.")] = v
    state_dict = corrected

torch.save(state_dict, dst)
print("Saved clean state_dict to:", dst)
print("File exists?", os.path.exists(dst))


Loading: /content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pt
Found 'model_state_dict' in checkpoint, unwrapping it.
Saved clean state_dict to: /content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pth
File exists? True


In [5]:
# Setup: imports and utility paths
import os
import torch
from transformers import AutoTokenizer

import gpt  # uses this repo's GPT implementation

# Prefer GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


In [6]:
# Configuration: set your checkpoint path and model config here
# NOTE: Adjust to match the model you trained.

# Example default paths (edit as needed)
# - Pretraining outputs typically saved by pretrain_gpt.py
# - SFT outputs typically saved by sft_gpt.py
 #= "/shared/0/projects/teaching/eecs595/models/pico-gpt/pretrained-models/"  # <- change me
checkpoint_path = '/content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pth'

# Model architecture used during training
config = {
    "vocab_size": None,          # will be computed from tokenizer below
    "context_length": 1024,
    "emb_dim": 512,
    "n_heads": 8,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": False,
}

# Generation defaults (you can tweak later)
max_new_tokens = 128
temperature = 0.8
print(checkpoint_path)

/content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pth


In [7]:
# Tokenizer: use repo's setup to ensure special tokens match training
# This mirrors `gpt.setup_tokenizer()` behavior.

tokenizer = gpt.setup_tokenizer()

# Compute actual vocab size that includes special tokens used during training
special_tokens = ["<|user|>", "<|assistant|>", "<|end|>", "<|system|>", "<|pad|>"]
max_token_id = max(tokenizer.convert_tokens_to_ids(tok) for tok in special_tokens)
actual_vocab_size = max_token_id + 1
print(f"Tokenizer vocab size detected: {actual_vocab_size}")

if config["vocab_size"] is None:
    config["vocab_size"] = actual_vocab_size


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer vocab size detected: 50262


In [8]:
# Load model and checkpoint

# Create model in CPU first to avoid GPU OOM on load
model = gpt.GPTModel(config)

print(checkpoint_path)
# Load checkpoint state dict (map to CPU, then move)
state_dict = torch.load(checkpoint_path, map_location='cpu')

# Backward-compat: fix a known typo from some checkpoints
if 'embbeding.token_embeddings.weight' in state_dict:
    corrected = {}
    for k, v in state_dict.items():
        corrected[k.replace('embbeding.', 'embedding.')] = v
    state_dict = corrected

# Validate vocab size compatibility with checkpoint
original_vocab_size = state_dict['embedding.token_embeddings.weight'].shape[0]
if original_vocab_size != config['vocab_size']:
    raise ValueError(f"Vocabulary size mismatch: checkpoint={original_vocab_size}, expected={config['vocab_size']}")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Loaded state_dict. Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")

# Move model to device
model = model.to(device)
model.eval()

print("Model loaded successfully!")


/content/drive/MyDrive/gpt_model/checkpoint_epoch_1.pth
Loaded state_dict. Missing keys: 0, Unexpected keys: 0
Model loaded successfully!


In [9]:
# Simple generation helper using repo's functions

def generate(prompt: str, max_new_tokens: int = None, temperature: float = None):
    max_tokens = max_new_tokens if max_new_tokens is not None else globals().get('max_new_tokens', 128)
    temp = temperature if temperature is not None else globals().get('temperature', 1.0)
    with torch.no_grad():
        model_device = next(model.parameters()).device
        encoded = tokenizer.encode(prompt)
        input_ids = torch.tensor(encoded, dtype=torch.long, device=model_device).unsqueeze(0)
        out_ids = gpt.generate_new_tokens(
            model=model,
            idx=input_ids,
            max_new_tokens=max_tokens,
            context_size=config['context_length'],
            temperature=temp,
        )
        # Move to CPU for decoding only
        out_ids = out_ids.to('cpu')
        return tokenizer.decode(out_ids.squeeze(0).tolist())

# Quick smoke test
print(generate("Hello, my name is", max_new_tokens=32, temperature=0.9))


Hello, my name is HIEK. From city C, a long line between A.D. 1693 and A.D. 1690 to Mr. 69, a Wes


In [34]:
print(generate("Sunsets are", 64, 1))


Sunsets are scary creatures that can also be more aggressive or stand tall than humans. Frequently mentioned can include clouds, lake trunks, mountains, sharks, sea lions, pink whales, penguins, swans, rye, and wild sirens. Just remember if you've got your life changing.
Many cockroaches consider


In [35]:
print(generate("Autumn is ", 64, 0.8))


Autumn is vernacular. It’s a hard time to fall asleep in the morning because the bell plays and is cold. It can also get hot sleep.
When do people get some night’s sleep? How do you get good sleep?
However, it is always a good idea to get up to sleep


In [33]:
print(generate("The waves in the ocean are", 32, 0.7))


The waves in the ocean are massive, but solid and rock at high altitude, low-oxygen exposures increase affect soils as they expand
On close inspection, marine marshes and Squall


In [18]:
print(generate("In a surprising discovery, researchers found that", 50, 0.9))


In a surprising discovery, researchers found that the gene variants of the Gamma Spectigmatin Complex (RTSO) have lowered serum bilirubin levels in the neurodegenerative state. These levels have been associated with epilepsy, cognitive impairment, hearing loss, and even cognitive dystro


In [29]:
print(generate("Machine Learning is a field of computer science that", 64, 0.8))


Machine Learning is a field of computer science that aims to revolutionize the way we live. Everything from computers to computers to digital cars, to health care, to health care, to living environments where there is no building blocks for all people.
A new generation of computer scientists is using real-world data to detect human disease and pathogenesis. This extra data can


In [30]:
chat_prompt = (
    "<|system|>You are a helpful assistant.<|end|>\n"
    "<|user|>Write a haiku about autumn leaves.<|end|>\n"
    "<|assistant|>"
)

print(generate(chat_prompt, max_new_tokens=40, temperature=0.8))


<|system|>You are a helpful assistant.<|end|>
<|user|>Write a haiku about autumn leaves.<|end|>
<|assistant|>Write someone who has had to take a haiku to a new place.
Often, you may hear that your line is a start or end, if you give an projection of your relationship with the


In [31]:
chat_prompt = (
    "<|system|>You are a helpful assistant.<|end|>\n"
    "<|user|>Explain overfitting in one sentence.<|end|>\n"
    "<|assistant|>"
)

print(generate(chat_prompt, max_new_tokens=40, temperature=0.7))


<|system|>You are a helpful assistant.<|end|>
<|user|>Explain overfitting in one sentence.<|end|>
<|assistant|>: Write a letter.hog.exe
route: Write a letter.hog.exe
If you are a seasoned novice, you must write a letter.hog.exe
ong-in:


## Chat-style prompting (optional)
You can format prompts with the special tokens used in training for chat-like behavior. Example:

```
<|system|>You are a helpful assistant.<|end|>
<|user|>Write a haiku about autumn leaves.<|end|>
<|assistant|>
```

Then call `generate(prompt)`; the model will continue the assistant response.
